In [ ]:
!pip install dspy-ai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.4/312.4 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.1/41.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 12.6 MB/s eta 0:00:00


In [ ]:
import dspy
import random
from dspy import LM
import pandas as pd
from datasets import load_dataset
from getpass import getpass
import os

# Enter your Groq API key securely
os.environ["GROQ_API_KEY"] = getpass("Enter GROQ API Key: ")
groq_lm = LM(model="groq/llama-3.1-8b-instant")



# Configure DSPy
dspy.settings.configure(
    lm=groq_lm,

)



Enter GROQ API Key: ··········


In [ ]:
pip install datasets pandas scikit-learn

In [ ]:
import pandas as pd
from datasets import load_dataset

# Load dataset (correct version)
dataset = load_dataset("clinc_oos", "plus")
intent_names = dataset["train"].features["intent"].names

print(dataset)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plus/train-00000-of-00001.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

plus/validation-00000-of-00001.parquet:   0%|          | 0.00/77.8k [00:00<?, ?B/s]

plus/test-00000-of-00001.parquet:   0%|          | 0.00/136k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15250 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'intent'],
        num_rows: 15250
    })
    validation: Dataset({
        features: ['text', 'intent'],
        num_rows: 3100
    })
    test: Dataset({
        features: ['text', 'intent'],
        num_rows: 5500
    })
})


In [ ]:
df_train = dataset["train"].to_pandas()
df_test = dataset["test"].to_pandas()

small_test = df_test.head(100)

print(df_train.head())

                                                text  intent
0  what expression would i use to say i love you ...      61
1  can you tell me how to say 'i do not speak muc...      61
2  what is the equivalent of, 'life is good' in f...      61
3  tell me how to say, 'it is a beautiful morning...      61
4  if i were mongolian, how would i say that i am...      61


In [ ]:
# Convert numeric labels into readable intent names
df_train["intent"] = df_train["intent"].apply(lambda x: intent_names[x])
df_test["intent"] = df_test["intent"].apply(lambda x: intent_names[x])

In [ ]:

df_train.iloc[0]

,0
text,what expression would i use to say i love you ...
intent,translate


In [ ]:
## run prediction
first_row = df_train.iloc[0]
print(f"customer message: {first_row['text']},real class: {first_row['intent']}")


customer message: what expression would i use to say i love you if i were an italian,real class: translate


In [ ]:
import random
import dspy

# Get unique labels from dataframe
labels = df_train["intent"].unique().tolist()
labels_str = ", ".join(labels)

def get_dspy_examples(df, k):
    dspy_examples = []

    labels = df["intent"].unique().tolist()

    for label in labels:
        label_df = df[df["intent"] == label].sample(n=k, random_state=42)

        for _, row in label_df.iterrows():
            dspy_examples.append(
                dspy.Example(
                    customer_message=row["text"],
                    answer=row["intent"]
                ).with_inputs("customer_message")
            )

    return dspy_examples
train_examples = get_dspy_examples(df_train, k=2)
all_test_examples = get_dspy_examples(df_test, k=10)

dev_examples = random.sample(all_test_examples, len(all_test_examples) // 2)
test_examples = [e for e in all_test_examples if e not in dev_examples]

In [ ]:
import dspy

class IntentSignature(dspy.Signature):
    """
    You are an intent classification system.

    Possible labels:
    translate
    transfer
    balance
    greeting
    complaint
    goodbye

    Output ONLY one label from the list above.
    Do NOT explain.
    Do NOT answer the question.
    """

    customer_message = dspy.InputField()
    answer = dspy.OutputField(desc="One label only.")

In [ ]:
# Baseline predictor
baseline_program = dspy.Predict(IntentSignature)

# Chain-of-thought predictor (needed for Bootstrap)
cot_program = dspy.ChainOfThought(IntentSignature)

In [ ]:
import random

def get_dspy_examples(df, k):
    dspy_examples = []

    labels = df["intent"].unique().tolist()

    for label in labels:
        label_df = df[df["intent"] == label].sample(n=k, random_state=42)

        for _, row in label_df.iterrows():
            dspy_examples.append(
                dspy.Example(
                    customer_message=row["text"],
                    answer=row["intent"]
                ).with_inputs("customer_message")
            )

    return dspy_examples


# Create train and test examples
train_examples = get_dspy_examples(df_train, k=2)
test_examples = get_dspy_examples(df_test, k=2)

In [ ]:
print("Baseline Predictions:\n")

for ex in test_examples[:3]:
    pred = baseline_program(customer_message=ex.customer_message)
    print("Message:", ex.customer_message)
    print("True:", ex.answer)
    print("Predicted:", pred.answer)
    print("------")

Baseline Predictions:

Message: how do you say dog in spanish
True: translate
Predicted: translate
------
Message: how would i say what is your name if i were french
True: translate
Predicted: translate
------
Message: make an immediate transfer of ten thousand from money market to checking
True: transfer
Predicted: transfer
------


In [ ]:
from dspy.teleprompt import LabeledFewShot

labeled_optimizer = LabeledFewShot(k=5)

labeled_model = labeled_optimizer.compile(
    baseline_program,
    trainset=train_examples
)

In [ ]:
def metric(example, prediction, trace=None):
    return prediction.answer.strip().lower() == example.answer.strip().lower()

In [ ]:
print("Testing LabeledFewShot:\n")

for ex in test_examples[:3]:
    pred = labeled_model(**ex.inputs())
    print("Message:", ex.customer_message)
    print("True:", ex.answer)
    print("Predicted:", pred.answer)
    print("------")

Testing LabeledFewShot:

Message: how do you say dog in spanish
True: translate
Predicted: translate
------
Message: how would i say what is your name if i were french
True: translate
Predicted: balance
------
Message: make an immediate transfer of ten thousand from money market to checking
True: transfer
Predicted: transfer
------


In [ ]:
from dspy.teleprompt import BootstrapFewShot

bootstrap_optimizer = BootstrapFewShot(metric=metric)

optimized_model = bootstrap_optimizer.compile(
    cot_program,
    trainset=train_examples
)

  1%|          | 3/302 [00:01<02:34,  1.93it/s]2026/03/03 13:39:50 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'customer_message': 'i need to move money from one account to another', 'answer': 'transfer'}) (input_keys={'customer_message'}) with <function metric at 0x7a6fc01d53a0> due to litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01khgk8a2ffew8evrmceyyqwy3` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5055, Requested 1339. Please try again in 3.94s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}
.
  2%|▏         | 5/302 [00:13<18:30,  3.74s/it]2026/03/03 13:40:01 ERROR dspy.teleprompt.bootstrap: Failed to run or to evaluate example Example({'customer_message': "i'll need you to set a timer for 10 minutes", 'answer': 'timer'}) (input_k

RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `llama-3.1-8b-instant` in organization `org_01khgk8a2ffew8evrmceyyqwy3` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 5285, Requested 1336. Please try again in 6.21s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing","type":"tokens","code":"rate_limit_exceeded"}}


In [ ]:
import time

def evaluate_model(model, examples, n=5):
    correct = 0

    for ex in examples[:n]:
        pred = model(customer_message=ex.customer_message)

        predicted = pred.answer.strip().lower()
        true = ex.answer.strip().lower()

        print("Message:", ex.customer_message)
        print("True:", true)
        print("Predicted:", predicted)
        print("------")

        if predicted == true:
            correct += 1

        time.sleep(2)  # prevent rate limit

    return correct / n

In [ ]:
for ex in test_examples[:3]:
    pred = baseline_program(customer_message=ex.customer_message)
    print("TRUE:", ex.answer)
    print("PRED:", pred.answer)
    print("------")

In [ ]:
baseline_acc = evaluate_model(baseline_program, test_examples, n=3)
labeled_acc = evaluate_model(labeled_model, test_examples, n=3)
bootstrap_acc = evaluate_model(optimized_model, test_examples, n=3)

print("\nFINAL COMPARISON")
print("Baseline:", baseline_acc)
print("Labeled FewShot:", labeled_acc)
print("Bootstrap FewShot:", bootstrap_acc)

In [ ]:
user_text = input("Enter customer message: ")

prediction = optimized_model(customer_message=user_text)

print("Predicted Intent:", prediction.answer)